---
# `Wikipedia Retriever and Vector Store Retriever`
---

### Introduction
- First type of Retriever
- Based on Data Source- choose the retrieve
- For wikipedia : use Wikipedia Retriever
- for Vector : use Vector Store Retriever

### Wikipedia Retriever
- Data source  is WikiPedia
- Wikipedia --> Retriever | query --> Retriever --> Wikipedia API --> 
- a wikipedia retriever is a retriever that fetches the Relevant content from the wikipedia api from a given query

### Vector Store Retriever
- A Langchain Vector Store Retriever is the Most Common Type of Retriever that search and fetch the Most Relevant Documents from a vector store based on Semantic Similiarity using vector embeddings

Similarity Search: basic idea to fetch the relevant docs

### How it Works
- You store your doucments in a Vector Store like FAISS, ChromaDB, Weaviate, AstraDB etc
- Each doc is converted into a Dense Vector using an Embedding Vector
- WHen the user enters a Query , it's also turned into a Vector
- The Retrievers compares the Query Vector with the Store Vectors
- It Retrievers the Top K most similar results

# `Detailed Notes 1`

# Wikipedia Retriever vs Vector Store Retriever in LangChain

Both are **Retrievers**, but they get information from completely different sources.

The easiest way to remember:

> **Wikipedia Retriever → searches Wikipedia**
> **Vector Store Retriever → searches your own embedded knowledge base**

---

# 1. What is a Retriever?

A Retriever takes a user's query and returns relevant documents.

```text
User Query
    ↓
Retriever
    ↓
Relevant Documents
    ↓
LLM
    ↓
Answer
```

For example:

```text
Question:
"What is Retrieval-Augmented Generation?"

        ↓

Retriever

        ↓

Relevant Documents
```

The Retriever **retrieves information**; it doesn't generate the final answer.

---

# 2. Wikipedia Retriever

## Definition

> **Wikipedia Retriever is a LangChain retriever that searches Wikipedia for information related to a user's query.**

Instead of creating your own vector database, you can use Wikipedia as an external knowledge source.

Architecture:

```text
User Query
     ↓
Wikipedia Retriever
     ↓
Wikipedia
     ↓
Relevant Articles
     ↓
LLM
     ↓
Answer
```

---

# 3. Why Use Wikipedia Retriever?

Suppose the user asks:

> "Who was Alan Turing?"

Instead of having your own documents:

```text
Your Documents
      ↓
Retriever
```

you can search Wikipedia:

```text
Question
   ↓
Wikipedia Retriever
   ↓
Wikipedia
   ↓
Alan Turing article
   ↓
LLM
```

Useful for:

* General knowledge
* Historical information
* People
* Places
* Scientific concepts
* Companies
* Events

---

# 4. Install Wikipedia Integration

In modern LangChain setups, the Wikipedia integration is provided separately.

```bash
pip install -U langchain-community wikipedia
```

---

# 5. Import Wikipedia Retriever

```python
from langchain_community.retrievers import WikipediaRetriever
```

Create it:

```python
retriever = WikipediaRetriever(
    top_k_results=3,
    doc_content_chars_max=4000
)
```

Here:

### `top_k_results`

```python
top_k_results=3
```

means:

> Retrieve up to 3 Wikipedia results.

### `doc_content_chars_max`

```python
doc_content_chars_max=4000
```

limits the amount of document content returned.

---

# 6. Invoke Wikipedia Retriever

```python
docs = retriever.invoke(
    "Alan Turing"
)
```

Print the results:

```python
for doc in docs:
    print(doc.page_content)
    print(doc.metadata)
```

The returned documents generally contain content plus metadata such as source information.

---

# 7. Complete Wikipedia Example

```python
from langchain_community.retrievers import WikipediaRetriever

retriever = WikipediaRetriever(
    top_k_results=3,
    doc_content_chars_max=4000
)

query = "Artificial Intelligence"

docs = retriever.invoke(query)

for i, doc in enumerate(docs, start=1):
    print(f"\n--- Document {i} ---")
    print(doc.page_content[:1000])
    print(doc.metadata)
```

Conceptually:

```text
"Artificial Intelligence"
          ↓
Wikipedia Retriever
          ↓
Wikipedia Search
          ↓
Relevant Articles
          ↓
LangChain Documents
```

---

# 8. Wikipedia Retriever + LLM

You can combine the Retriever with an LLM.

```python
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4.1-mini"
)
```

Retrieve:

```python
docs = retriever.invoke(
    "Who invented the World Wide Web?"
)
```

Create context:

```python
context = "\n\n".join(
    doc.page_content for doc in docs
)
```

Then:

```python
prompt = f"""
Answer the question using the following Wikipedia context.

Context:
{context}

Question:
Who invented the World Wide Web?
"""

response = llm.invoke(prompt)

print(response.content)
```

Architecture:

```text
User Question
      ↓
Wikipedia Retriever
      ↓
Wikipedia
      ↓
Relevant Articles
      ↓
Context
      ↓
LLM
      ↓
Answer
```

---

# 9. What is Vector Store Retriever?

Now let's look at the second type.

> **A Vector Store Retriever retrieves relevant documents from a vector store using vector similarity search.**

This is extremely important for **RAG applications**.

Architecture:

```text
Your Documents
      ↓
Document Loader
      ↓
Text Splitter
      ↓
Embedding Model
      ↓
Vector Store
      ↓
Vector Store Retriever
      ↓
Relevant Chunks
      ↓
LLM
```

---

# 10. Why Use Vector Store Retriever?

Suppose you have:

```text
Company Documentation
       ↓
100 PDFs
       ↓
50,000 chunks
```

You want users to ask:

> "How do I reset my password?"

You don't want to search the entire documents manually.

Instead:

```text
Question
   ↓
Embedding
   ↓
Vector Search
   ↓
Relevant Chunks
   ↓
LLM
```

This is the typical architecture of a private/company RAG system.

---

# 11. Create a Vector Store

Let's use Chroma.

Install:

```bash
pip install -U langchain-chroma langchain-openai
```

Import:

```python
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
```

Create embedding model:

```python
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)
```

Create vector store:

```python
vector_store = Chroma(
    collection_name="my_documents",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)
```

---

# 12. Add Documents

```python
documents = [
    "RAG combines retrieval with language generation.",
    "Embeddings convert text into numerical vectors.",
    "Vector stores allow similarity search over embeddings.",
    "LangChain is a framework for building LLM applications."
]
```

Add them:

```python
vector_store.add_texts(documents)
```

Internally:

```text
Document
   ↓
Embedding Model
   ↓
Vector
   ↓
Chroma
```

---

# 13. Create Vector Store Retriever

```python
retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 2
    }
)
```

Now:

```text
Vector Store
     ↓
as_retriever()
     ↓
Retriever
```

---

# 14. Invoke the Retriever

```python
docs = retriever.invoke(
    "How does RAG work?"
)
```

Print:

```python
for doc in docs:
    print(doc.page_content)
```

Possible result:

```text
RAG combines retrieval with language generation.

Vector stores allow similarity search over embeddings.
```

---

# 15. How Vector Store Retriever Works

Suppose your database contains:

```text
Document A → Python
Document B → Machine Learning
Document C → RAG
Document D → LangChain
```

User asks:

```text
"What is Retrieval-Augmented Generation?"
```

The process is:

```text
Query
  ↓
Embedding Model
  ↓
Query Vector
  ↓
Vector Store
  ↓
Similarity Search
  ↓
Top-K Documents
```

For example:

```text
Document C → 0.94
Document D → 0.82
Document B → 0.51
Document A → 0.21
```

If:

```python
k = 2
```

the Retriever returns:

```text
Document C
Document D
```

---

# 16. Key Difference

This is the most important part.

| Feature                  | Wikipedia Retriever             | Vector Store Retriever                |
| ------------------------ | ------------------------------- | ------------------------------------- |
| Data source              | Wikipedia                       | Your vector store                     |
| Search mechanism         | Wikipedia search                | Vector similarity / backend retrieval |
| Your own documents       | ❌                               | ✅                                     |
| Requires embeddings      | Not necessarily                 | Usually yes                           |
| Requires vector DB/store | ❌                               | ✅                                     |
| Best for                 | General knowledge               | Private/custom knowledge              |
| Typical use              | External information            | RAG                                   |
| Offline                  | ❌ Generally requires web access | ✅ Can be local                        |
| Data control             | Limited                         | High                                  |
| Example                  | "Who was Einstein?"             | "What does my company policy say?"    |

---

# 17. Simple Example

## Wikipedia Retriever

Question:

> "Who was Albert Einstein?"

```text
Question
   ↓
Wikipedia Retriever
   ↓
Wikipedia
   ↓
Einstein article
   ↓
LLM
```

---

## Vector Store Retriever

Question:

> "What is our company's leave policy?"

```text
Question
   ↓
Vector Store Retriever
   ↓
Company Documents
   ↓
Relevant Policy Chunk
   ↓
LLM
```

This distinction is extremely important in interviews.

---

# 18. Wikipedia Retriever Does Not Need Your Vector Database

Wikipedia Retriever:

```text
User Query
     ↓
Wikipedia API/Search
     ↓
Wikipedia Documents
```

Vector Store Retriever:

```text
User Query
     ↓
Embedding
     ↓
Vector Store
     ↓
Similarity Search
```

So:

> **Wikipedia Retriever retrieves from an external knowledge source, while Vector Store Retriever retrieves from your indexed knowledge base.**

---

# 19. Wikipedia Retriever vs RAG

Wikipedia Retriever can actually be used as part of a RAG-style pipeline.

```text
User Question
      ↓
Wikipedia Retriever
      ↓
Wikipedia Context
      ↓
Prompt
      ↓
LLM
      ↓
Answer
```

This is still retrieval + generation.

But the retrieval source is:

```text
Wikipedia
```

instead of:

```text
Chroma / Qdrant / Pinecone / etc.
```

---

# 20. Vector Store Retriever in RAG

This is the more common architecture for private knowledge.

```text
             KNOWLEDGE BASE
                   │
                   ▼
             Document Loader
                   │
                   ▼
              Text Splitter
                   │
                   ▼
             Embedding Model
                   │
                   ▼
              Vector Store
                   │
                   │
                   ▼
              Retriever
                   ▲
                   │
               User Query
                   │
                   ▼
            Relevant Chunks
                   │
                   ▼
                  LLM
                   │
                   ▼
                Answer
```

---

# 21. When Should You Use Wikipedia Retriever?

Use it when:

* You need general knowledge.
* Information is available on Wikipedia.
* You don't need private/company documents.
* You want a quick prototype.
* You want to demonstrate external retrieval.

Example project:

> **Wikipedia Research Assistant**

```text
User
 ↓
Wikipedia Retriever
 ↓
Relevant Articles
 ↓
LLM
 ↓
Summary
```

---

# 22. When Should You Use Vector Store Retriever?

Use it when:

* You have PDFs.
* You have company documentation.
* You have websites you have indexed.
* You have research papers.
* You have product documentation.
* You have private knowledge.
* You're building a RAG application.

Example:

> **Company Knowledge Assistant**

```text
Company PDFs
     ↓
PDF Loader
     ↓
Text Splitter
     ↓
Embeddings
     ↓
Qdrant
     ↓
Retriever
     ↓
LLM
```

---

# 23. Important: Retriever Is an Abstraction

One of the most important LangChain concepts is:

> **Retriever is not necessarily a database.**

It is an interface/component for retrieving documents.

Different implementations can retrieve from different sources:

```text
                    Retriever
                       │
       ┌───────────────┼────────────────┐
       ↓               ↓                ↓
   Wikipedia       Vector Store      Keyword
   Retriever        Retriever        Retriever
       │               │                │
       ↓               ↓                ↓
  Wikipedia       Chroma/Qdrant       BM25
```

This is why LangChain's Retriever abstraction is useful.

---

# 24. Interview Questions

## Beginner

### Q1. What is a Wikipedia Retriever?

> It is a LangChain retriever that searches Wikipedia and returns relevant documents for a query.

### Q2. What is a Vector Store Retriever?

> It retrieves relevant documents from a vector store, typically using embedding-based similarity search.

### Q3. What is the biggest difference?

> Wikipedia Retriever searches an external Wikipedia knowledge source, while Vector Store Retriever searches your own indexed vector data.

---

# 25. Intermediate

### Q4. Does Wikipedia Retriever use embeddings?

> Not necessarily. It primarily uses Wikipedia's search/retrieval mechanism rather than requiring you to embed your entire Wikipedia knowledge base into a vector store.

### Q5. Does Vector Store Retriever require embeddings?

> For semantic vector retrieval, yes. Documents and queries need to be represented as vectors using an embedding model.

### Q6. Which one is better for private company documents?

> Vector Store Retriever, because you control the documents, embeddings, metadata, indexing, and retrieval pipeline.

---

# 26. Scenario-Based Questions

### Q7. You are building a chatbot over 5,000 company PDFs. Which Retriever would you use?

**Answer:**

> A Vector Store Retriever.

Architecture:

```text
PDFs
 ↓
PDF Loader
 ↓
Text Splitter
 ↓
Embedding Model
 ↓
Vector Store
 ↓
Retriever
 ↓
LLM
```

---

### Q8. You want to build a chatbot that answers questions about historical personalities using Wikipedia. Which would you choose?

**Answer:**

> Wikipedia Retriever can be a simple choice because the knowledge source is Wikipedia and you don't need to maintain your own vector database for the basic use case.

---

### Q9. Your vector retriever gives poor results. What would you check?

```text
1. Chunk size
2. Chunk overlap
3. Embedding model
4. Top-K
5. Metadata filtering
6. Similarity metric
7. Query quality
8. Hybrid search
9. Reranking
```

---

# 27. 30-Second Revision

```text
Wikipedia Retriever
        ↓
Searches Wikipedia
        ↓
Returns Wikipedia documents
```

```text
Vector Store Retriever
        ↓
Searches your vector store
        ↓
Returns relevant documents/chunks
```

### Remember:

> **Wikipedia Retriever = External knowledge**

> **Vector Store Retriever = Your indexed knowledge**

---

# 28. 2-Minute Revision

## Wikipedia Retriever

```text
Query
 ↓
Wikipedia
 ↓
Relevant Articles
```

Best for:

```text
General knowledge
Historical information
People
Places
Scientific topics
```

---

## Vector Store Retriever

```text
Query
 ↓
Embedding
 ↓
Vector Store
 ↓
Similarity Search
 ↓
Relevant Chunks
```

Best for:

```text
PDF RAG
Company Knowledge Base
Private Documents
Research Papers
Documentation
```

### LangChain Code

**Wikipedia:**

```python
from langchain_community.retrievers import WikipediaRetriever

retriever = WikipediaRetriever(
    top_k_results=3
)

docs = retriever.invoke("Albert Einstein")
```

**Vector Store:**

```python
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

docs = retriever.invoke(
    "What is RAG?"
)
```

### Golden Interview Answer

> **Both Wikipedia Retriever and Vector Store Retriever implement the Retriever concept in LangChain, but their data sources are different. Wikipedia Retriever retrieves information from Wikipedia, while Vector Store Retriever retrieves relevant documents from an indexed vector store, typically using embeddings and similarity search.**


# `Detailed Notes 2`

# Wikipedia Retriever in LangChain

**Wikipedia Retriever** is a LangChain retriever that allows an LLM application to **search Wikipedia and retrieve relevant articles/content based on a user's query**.

It is useful when you want your application to retrieve **general-world knowledge** from Wikipedia instead of searching your own vector database.

---

## 1. What is a Wikipedia Retriever?

Normally, in a RAG application you might have:

```text
Your PDF
   ↓
Chunking
   ↓
Embeddings
   ↓
Chroma / FAISS
   ↓
Retriever
```

With a Wikipedia Retriever:

```text
User Question
      ↓
Wikipedia Retriever
      ↓
Wikipedia Search
      ↓
Relevant Wikipedia Pages
      ↓
Documents
      ↓
LLM
      ↓
Answer
```

So the important difference is:

> **A Wikipedia Retriever retrieves information from Wikipedia rather than from your own vector database.**

---

# 2. Why Use Wikipedia Retriever?

Imagine your user asks:

> "Who was Alan Turing?"

If your application has no information about Alan Turing in its local database, Chroma or FAISS cannot help.

But Wikipedia contains information about him.

So:

```text
Question
   ↓
Wikipedia Retriever
   ↓
Wikipedia
   ↓
Alan Turing article
   ↓
Relevant content
   ↓
LLM
```

This is useful for applications requiring **general factual knowledge**.

---

# 3. Wikipedia Retriever vs Vector Store Retriever

This is important because you just learned vector-store retrievers.

### Vector Store Retriever

```text
Your Documents
      ↓
Embeddings
      ↓
Vector Database
      ↓
Retriever
```

Examples:

```text
PDF
Company documents
Research papers
Product documentation
Private knowledge base
```

### Wikipedia Retriever

```text
User Query
      ↓
Wikipedia Retriever
      ↓
Wikipedia
      ↓
Relevant Pages
```

So:

| Feature             | Vector Store Retriever   | Wikipedia Retriever |
| ------------------- | ------------------------ | ------------------- |
| Data source         | Your stored knowledge    | Wikipedia           |
| Embeddings required | Usually yes              | Not necessarily     |
| Vector DB required  | Usually                  | No                  |
| Best for            | Private/domain knowledge | General knowledge   |
| Example             | Company policies         | Historical facts    |
| RAG                 | Yes                      | Yes                 |

---

# 4. Architecture

A simple Wikipedia-based RAG system:

```text
                   USER
                     │
                     ▼
                  Question
                     │
                     ▼
            Wikipedia Retriever
                     │
                     ▼
              Wikipedia Search
                     │
                     ▼
              Relevant Articles
                     │
                     ▼
                 Documents
                     │
                     ▼
                   Prompt
                     │
                     ▼
                    LLM
                     │
                     ▼
                  Answer
```

---

# 5. Installation

Depending on the LangChain version and integration package you're using, you'll typically install the Wikipedia integration:

```bash
pip install langchain-community wikipedia
```

The `wikipedia` package provides Python access to Wikipedia, while LangChain provides the retriever abstraction.

---

# 6. Import Wikipedia Retriever

A common LangChain implementation is:

```python
from langchain_community.retrievers import WikipediaRetriever
```

Then:

```python
retriever = WikipediaRetriever(
    top_k_results=3,
    doc_content_chars_max=4000
)
```

Here:

### `top_k_results`

Controls approximately how many Wikipedia results to retrieve.

```python
top_k_results=3
```

means:

```text
Query
 ↓
Wikipedia Search
 ↓
Top 3 results
```

### `doc_content_chars_max`

Limits the amount of content returned for each document.

---

# 7. Basic Example

Let's search for:

```text
"Artificial Intelligence"
```

```python
from langchain_community.retrievers import WikipediaRetriever

retriever = WikipediaRetriever(
    top_k_results=3,
    doc_content_chars_max=4000
)

docs = retriever.invoke(
    "Artificial Intelligence"
)

for doc in docs:
    print(doc.page_content)
```

The returned objects are typically LangChain `Document` objects.

---

# 8. What Does It Return?

A returned document generally looks conceptually like:

```python
Document(
    page_content="Artificial intelligence (AI) is...",
    metadata={
        "title": "Artificial intelligence",
        "summary": "...",
        "source": "..."
    }
)
```

The exact metadata can vary by integration/version.

The important structure is:

```text
Document
├── page_content
└── metadata
```

This is the same `Document` abstraction you've seen with other LangChain retrievers.

---

# 9. Example

Suppose you run:

```python
docs = retriever.invoke(
    "Albert Einstein"
)
```

You might retrieve documents such as:

```text
Document 1
Title: Albert Einstein
Content: Albert Einstein was a physicist...

Document 2
Title: Theory of Relativity
Content: The theory of relativity...

Document 3
Title: General Relativity
Content: General relativity is...
```

The Retriever gives these documents to your application.

It **doesn't generate the final answer**.

---

# 10. Wikipedia Retriever + LLM

Now we can connect it to an LLM.

```text
User:
"Explain Albert Einstein's contribution to physics."
              │
              ▼
      Wikipedia Retriever
              │
              ▼
       Wikipedia Articles
              │
              ▼
           Context
              │
              ▼
             LLM
              │
              ▼
          Final Answer
```

The LLM uses the retrieved Wikipedia content as context.

---

# 11. Simple RAG Example

Conceptually:

```python
from langchain_community.retrievers import WikipediaRetriever
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

retriever = WikipediaRetriever(
    top_k_results=3,
    doc_content_chars_max=4000
)

model = ChatOpenAI(
    model="gpt-4.1-mini"
)

prompt = ChatPromptTemplate.from_template("""
Answer the question using the context below.

Context:
{context}

Question:
{question}
""")

question = "Who was Alan Turing?"

docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content for doc in docs
)

response = model.invoke(
    prompt.format(
        context=context,
        question=question
    )
)

print(response.content)
```

The workflow is:

```text
Question
   ↓
WikipediaRetriever
   ↓
Wikipedia
   ↓
Documents
   ↓
Context
   ↓
Prompt
   ↓
LLM
   ↓
Answer
```

---

# 12. Does Wikipedia Retriever Use Embeddings?

This is a key difference.

A typical vector retriever works like:

```text
Question
   ↓
Embedding Model
   ↓
Query Vector
   ↓
Vector Database
   ↓
Similarity Search
```

WikipediaRetriever instead performs a **Wikipedia search** and returns relevant pages.

Conceptually:

```text
Question
   ↓
Wikipedia Search
   ↓
Relevant Pages
   ↓
Documents
```

Therefore, you don't need to first build your own:

```text
Embedding
+
Chroma
+
FAISS
```

for this retriever.

---

# 13. Wikipedia Retriever vs FAISS

### FAISS

You need:

```text
Documents
 ↓
Embedding Model
 ↓
Vectors
 ↓
FAISS
```

Then:

```text
Query
 ↓
Embedding
 ↓
FAISS Search
```

### Wikipedia Retriever

```text
Query
 ↓
Wikipedia Search
 ↓
Wikipedia Documents
```

So:

```text
FAISS
→ searches your vector index

Wikipedia Retriever
→ searches Wikipedia
```

---

# 14. Wikipedia Retriever vs Chroma Retriever

### Chroma

Suppose you have:

```text
company_policy.pdf
```

You create:

```text
Chunks
 ↓
Embeddings
 ↓
Chroma
```

Then:

```text
User Query
 ↓
Chroma Retriever
 ↓
Company Policy
```

### Wikipedia

You don't need to ingest the company document.

Instead:

```text
User Query
 ↓
Wikipedia Retriever
 ↓
Wikipedia
```

Therefore:

> **Chroma is appropriate when you control the knowledge base; Wikipedia Retriever is useful when Wikipedia itself is the knowledge source.**

---

# 15. Wikipedia Retriever in RAG

There are actually two common RAG patterns.

### Pattern 1 — Private RAG

```text
Company Documents
       ↓
Vector DB
       ↓
Retriever
       ↓
LLM
```

Example:

> "What is our company's leave policy?"

---

### Pattern 2 — Wikipedia RAG

```text
Wikipedia
     ↓
Wikipedia Retriever
     ↓
Relevant Articles
     ↓
LLM
```

Example:

> "Explain the French Revolution."

---

# 16. Wikipedia Retriever vs Web Search

Don't confuse these.

### Wikipedia Retriever

Searches:

```text
Wikipedia
```

### Web Search Tool

Can search:

```text
Internet
├── Websites
├── News
├── Blogs
├── Documentation
└── Wikipedia
```

So Wikipedia Retriever is much narrower.

```text
Web Search
   │
   ├── Wikipedia
   ├── BBC
   ├── Reuters
   ├── Documentation
   └── Other websites
```

versus:

```text
Wikipedia Retriever
       │
       └── Wikipedia
```

---

# 17. Advantages

### 1. Very easy to use

You don't need to build a vector database first.

### 2. Good for general knowledge

Useful for topics such as:

* History
* Geography
* Science
* Technology
* Famous people
* Organizations
* Countries

### 3. Integrates with LangChain

It follows LangChain's Retriever abstraction.

### 4. Useful for learning RAG

It is a good way to understand:

```text
Retriever
   ↓
Documents
   ↓
Prompt
   ↓
LLM
```

without first dealing with:

```text
Chunking
Embeddings
Vector DB
Indexing
```

---

# 18. Limitations

### 1. Wikipedia Only

It cannot retrieve information from arbitrary websites.

### 2. Not a Private Knowledge Base

You cannot use it to search:

```text
Your company's internal documents
Private PDFs
Private databases
Internal GitHub repositories
```

For that, use an appropriate data connector or vector store/retriever.

### 3. Search Quality Depends on Wikipedia

If Wikipedia doesn't have good coverage of a topic, retrieval will be poor.

### 4. Not Real-Time General Web Search

Wikipedia content may not reflect the latest developments.

For breaking news or rapidly changing information, a dedicated web/news search system is more appropriate.

---

# 19. When Should You Use It?

Use Wikipedia Retriever when:

```text
You need
      ↓
General factual knowledge
      ↓
Wikipedia is an appropriate source
```

For example:

```text
"What is quantum mechanics?"
"Who was Nikola Tesla?"
"What is the history of the Internet?"
"Explain the French Revolution."
```

---

# 20. When Should You NOT Use It?

Don't use Wikipedia Retriever for:

```text
Company policies
Private PDFs
Customer records
Internal documentation
Real-time stock prices
Current news
Private databases
```

For example:

> "What is our company's reimbursement policy?"

Use:

```text
Company Documents
       ↓
Vector Store
       ↓
Retriever
```

not Wikipedia.

---

# 21. Important Concept: Retriever Abstraction

This is the bigger LangChain concept you should understand.

LangChain gives you a common interface:

```python
docs = retriever.invoke(query)
```

Different retrievers can implement that interface differently.

For example:

```text
                    Retriever
                       │
       ┌───────────────┼────────────────┐
       ▼               ▼                ▼
   Wikipedia         Chroma            BM25
   Retriever        Retriever         Retriever
       │               │                │
       ▼               ▼                ▼
   Wikipedia       Vector DB       Keyword Search
```

The application can work with:

```python
docs = retriever.invoke(query)
```

without needing to know all the underlying implementation details.

This is the power of an **abstraction**.

---

# 22. Interview Question

### What is Wikipedia Retriever in LangChain?

A good answer:

> **Wikipedia Retriever is a LangChain retriever that queries Wikipedia and returns relevant Wikipedia content as LangChain `Document` objects. It can be used as the retrieval component of a RAG pipeline without requiring you to create embeddings or maintain a vector database for the Wikipedia content.**

---

# 23. Compare the Retrievers You've Learned

You can now build this mental map:

| Retriever                 | Search Method              | Typical Use       |
| ------------------------- | -------------------------- | ----------------- |
| Vector Store Retriever    | Vector similarity          | RAG               |
| FAISS Retriever           | Vector similarity          | Local RAG         |
| Chroma Retriever          | Vector similarity          | Local/managed RAG |
| BM25 Retriever            | Keyword matching           | Exact terms       |
| Wikipedia Retriever       | Wikipedia search           | General knowledge |
| Multi-Query Retriever     | Multiple generated queries | Improve recall    |
| Parent Document Retriever | Child → parent retrieval   | Better context    |
| Ensemble Retriever        | Multiple retrieval methods | Hybrid retrieval  |

---

# 24. Final Mental Model

The most important thing is not the class name.

Understand **where the Retriever gets information from**:

```text
                     RETRIEVER
                         │
       ┌─────────────────┼─────────────────┐
       ▼                 ▼                 ▼
   Vector DB          Wikipedia          BM25
       │                 │                 │
       ▼                 ▼                 ▼
Semantic Search     Wiki Search       Keyword Search
       │                 │                 │
       └─────────────────┼─────────────────┘
                         ▼
                    Documents
                         │
                         ▼
                       LLM
                         │
                         ▼
                      Answer
```

### In one sentence:

> **Wikipedia Retriever is a LangChain retrieval component that searches Wikipedia for a user's query and returns relevant content as documents, which can then be passed to an LLM as context in a RAG pipeline.**

For your LangChain roadmap, the next logical concept after this is **VectorStore Retriever vs Wikipedia Retriever vs MultiQuery Retriever vs Contextual Compression Retriever**, because that will show you how retrieval strategies evolve from simple search to production-grade RAG.
